# 1. Instalasi Library
Kita memerlukan `rank_bm25` untuk menghitung fitur BM25, serta `optuna` untuk Hyperparameter Optimization (HPO). Tensorflow/Keras akan digunakan untuk RNN.

In [18]:
pip install rank_bm25 optuna scikit-learn nltk tensorflow pandas numpy

Note: you may need to restart the kernel to use updated packages.


# 2. Import Libraries
Import seluruh library yang dibutuhkan untuk EDA, Preprocessing, Feature Engineering, dan Pemodelan.

In [24]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Bidirectional, GRU, Dense, Concatenate, Dropout, GlobalMaxPooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import optuna

# Download stopwords bahasa indonesia
nltk.download('stopwords')
try:
    stop_words = set(stopwords.words('indonesian'))
except:
    # Fallback if indonesian is not perfectly loaded
    stop_words = set(['yang', 'di', 'dan', 'itu', 'dengan', 'untuk', 'tidak', 'ini', 'dari', 'dalam', 'akan', 'pada', 'juga', 'saya', 'ke', 'karena', 'tersebut', 'bisa', 'ada', 'mereka', 'lebih', 'sudah', 'atau', 'saat', 'oleh', 'menjadi'])

[nltk_data] Downloading package stopwords to C:\Users\Ezra
[nltk_data]     Faira\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# 3. Exploratory Data Analysis & Data Cleaning
Sesuai dengan alur EDA yang Anda berikan, kita membersihkan duplikat dan *missing values*, serta melihat distribusi kelas.
File diasumsikan berada di folder `data/`.

In [26]:
def clean_and_analyze(train, test=None):
    print("\n=== 1. CEK DUPLIKAT ===")
    duplikat_awal = train.duplicated(subset=["title", "content"]).sum()
    print(f"Jumlah duplikat di Train : {duplikat_awal}")

    train = train.drop_duplicates(subset=["title", "content"], keep="first").reset_index(drop=True)
    print(f"Sisa duplikat di Train   : {train.duplicated(subset=['title', 'content']).sum()}")
    print("-" * 40)

    print("\n=== 2. CEK MISSING VALUES ===")
    train = train.dropna(subset=["title", "content", "label"]).reset_index(drop=True)
    
    if test is not None:
        test["title"] = test["title"].fillna("").astype(str)
        test["content"] = test["content"].fillna("").astype(str)

    print("-" * 40)

    print("\n=== 3. DISTRIBUSI KELAS (TRAIN) ===")
    label_counts = train["label"].value_counts()
    label_pct = train["label"].value_counts(normalize=True).mul(100).round(2)
    dist_df = pd.DataFrame({"Jumlah Baris": label_counts, "Persentase (%)": label_pct})
    print(dist_df)
    print("-" * 40)

    return train, test

# Load Data (Ganti path sesuai letak file lokal Anda saat running)
try:
    train_df = pd.read_csv('data/train.csv')
    # test_df = pd.read_csv('data/test.csv') # Uncomment jika file test sudah ada
    test_df = None
    train_df, test_df = clean_and_analyze(train_df, test_df)
except FileNotFoundError:
    print("Folder data/ atau file CSV belum ditemukan. Harap pastikan letak file sudah sesuai.")
    # Dummy dataframe for notebook execution validation if files are missing
    train_df = pd.DataFrame({'title': ['A', 'B'], 'content': ['C', 'D'], 'label': [1, 0]})


=== 1. CEK DUPLIKAT ===
Jumlah duplikat di Train : 3
Sisa duplikat di Train   : 0
----------------------------------------

=== 2. CEK MISSING VALUES ===
----------------------------------------

=== 3. DISTRIBUSI KELAS (TRAIN) ===
       Jumlah Baris  Persentase (%)
label                              
1             12960           90.04
0              1434            9.96
----------------------------------------


# 4. Labeling Data Eksternal (Handling Imbalance)
Distribusi label 0 sangat sedikit (9.96%). Untuk meningkatkan performa model (khususnya metrik Macro F1), kita akan menggunakan `news_data.csv`.
Strategi: 
1. Ambil baris dari `news_data.csv` yang memiliki judul dan isi (ini akan jadi kelas 1 / Sesuai).
2. Lakukan *shuffle* / acak isi beritanya agar tidak cocok dengan judulnya (ini akan jadi kelas 0 / Tidak Sesuai).
3. Gabungkan dengan data train utama agar kelas menjadi lebih seimbang.

In [27]:
def augment_with_external_data(train_df, external_path='data/news_data.csv', num_samples=8000):
    """Tambah data eksternal untuk mengatasi imbalance.
    Set sesuai kebutuhan: set USE_EXTERNAL = False untuk nonaktifkan."""
    try:
        news_df = pd.read_csv(external_path)
        news_df = news_df.dropna(subset=['title', 'article_text'])
        
        # Buat pasangan Positif (Label 1)
        pos_df = pd.DataFrame({
            'title': news_df['title'].values,
            'content': news_df['article_text'].values,
            'label': 1
        })
        
        # Buat pasangan Negatif (Label 0) dengan menukar/acak pasangannya
        shuffled_content = news_df['article_text'].sample(frac=1, random_state=42).reset_index(drop=True)
        neg_df = pd.DataFrame({
            'title': news_df['title'].values,
            'content': shuffled_content.values,
            'label': 0
        })
        
        # Rasio 1:1 agar tidak bias ke negatif
        n_pos = min(len(pos_df), num_samples // 2)
        n_neg = min(len(neg_df), num_samples // 2)
        aug_data = pd.concat([pos_df.head(n_pos), neg_df.head(n_neg)], ignore_index=True)
        
        combined_train = pd.concat([train_df, aug_data], ignore_index=True)
        print(f"\nDistribusi setelah augmentasi:")
        print(combined_train['label'].value_counts())
        return combined_train
    except FileNotFoundError:
        print("Data external tidak ditemukan, lewati tahap augmentasi.")
        return train_df

# SET KE False JIKA TIDAK PAKAI DATA EKSTERNAL
USE_EXTERNAL = False

if USE_EXTERNAL:
    train_df = augment_with_external_data(train_df, 'data/news_data.csv', num_samples=4000)
else:
    print("Augmentasi data eksternal dilewati.")



Distribusi kelas setelah penambahan data eksternal:
label
1    61.33
0    38.67
Name: proportion, dtype: float64


# 5. Text Preprocessing
Melakukan *lowercasing*, membersihkan tanda baca, dan menghapus *stopwords* (kata hubung) agar model dan ekstraksi fitur (TF-IDF & BM25) lebih fokus pada kata-kata penting.

In [28]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Hapus tanda baca dan angka
    text = re.sub(f'[{re.escape(string.punctuation)}0-9]', ' ', text)
    # Tokenisasi dan hapus stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    return " ".join(tokens)

print("Memulai Preprocessing...")
train_df['clean_title'] = train_df['title'].apply(preprocess_text)
train_df['clean_content'] = train_df['content'].apply(preprocess_text)
print("Preprocessing Selesai.")

Memulai Preprocessing...
Preprocessing Selesai.


# 6. Feature Engineering (Handcrafted Features)
Di sini kita mengekstrak kedekatan teks menggunakan **BM25**, **TF-IDF Cosine Similarity**, dan **Jaccard Similarity**.
Ini akan sangat membantu model dalam mendeteksi relevansi (tidak hanya bergantung pada Word Embedding internal RNN).

In [29]:
def calculate_jaccard(str1, str2):
    a = set(str1.split())
    b = set(str2.split())
    if not a or not b: return 0.0
    return len(a.intersection(b)) / len(a.union(b))

def extract_features(df, full_corpus_title=None, full_corpus_content=None):
    """Ekstrak fitur. full_corpus untuk TF-IDF fit (hindari leakage)."""
    print("Mengekstrak Fitur Kemiripan Teks...")
    
    # 1. Jaccard Similarity
    df['jaccard_sim'] = df.apply(lambda x: calculate_jaccard(x['clean_title'], x['clean_content']), axis=1)
    
    # 2. TF-IDF Cosine Similarity
    tfidf = TfidfVectorizer(max_features=5000)
    if full_corpus_title is not None:
        corpus_all = pd.concat([full_corpus_title, full_corpus_content])
    else:
        corpus_all = pd.concat([df['clean_title'], df['clean_content']])
    tfidf.fit(corpus_all)
    
    title_tfidf = tfidf.transform(df['clean_title'])
    content_tfidf = tfidf.transform(df['clean_content'])
    
    cosine_sim = np.array([cosine_similarity(t, c)[0][0] for t, c in zip(title_tfidf, content_tfidf)])
    df['tfidf_cosine_sim'] = cosine_sim
    
    # 3. BM25 Score (satu index untuk semua dokumen)
    tokenized_corpus = [doc.split() for doc in df['clean_content']]
    bm25 = BM25Okapi(tokenized_corpus)
    
    bm25_scores = []
    for idx, row in df.iterrows():
        query = row['clean_title'].split()
        if len(query) > 0:
            score = bm25.get_scores(query)[idx]
        else:
            score = 0.0
        bm25_scores.append(score)
    df['bm25_score'] = bm25_scores
    
    # 4. Perbedaan Panjang
    df['len_diff'] = df['clean_content'].apply(lambda x: len(x.split())) - df['clean_title'].apply(lambda x: len(x.split()))
    
    print("Feature Engineering Selesai.")
    return df, tfidf

# Simpan corpus train untuk fit TF-IDF saat proses test nanti
train_corpus_title = train_df['clean_title'].copy()
train_corpus_content = train_df['clean_content'].copy()

train_df, fitted_tfidf = extract_features(train_df)


Mengekstrak Fitur Kemiripan Teks...


TypeError: 'NoneType' object is not subscriptable

# 7. Persiapan Data untuk Jaringan Syaraf Tiruan (RNN)
Kita menggunakan arsitektur **Wide & Deep Learning**.
*   **Deep Component (RNN):** Menerima urutan *token/integer* teks, diproses lewat Keras `Embedding` layer yang dilatih dari awal, dan dipahami menggunakan **BiGRU** (Bidirectional Gated Recurrent Unit). *BiGRU dipilih karena komputasinya lebih ringan dari LSTM namun seringkali memberikan hasil yang setara atau lebih baik pada teks.*
*   **Wide Component (Dense):** Menerima fitur numerik yang baru saja kita ekstrak (BM25, Cosine, Jaccard).

In [ ]:
MAX_VOCAB = 25000
MAX_TITLE_LEN = 30
MAX_CONTENT_LEN = 300

# Tokenisasi untuk RNN
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(pd.concat([train_df['clean_title'], train_df['clean_content']]))

title_seq = pad_sequences(tokenizer.texts_to_sequences(train_df['clean_title']), maxlen=MAX_TITLE_LEN, padding='post')
content_seq = pad_sequences(tokenizer.texts_to_sequences(train_df['clean_content']), maxlen=MAX_CONTENT_LEN, padding='post')

# Siapkan fitur tabular numerik
tabular_features = train_df[['jaccard_sim', 'tfidf_cosine_sim', 'bm25_score', 'len_diff']].values
labels = train_df['label'].values

# Train-Test Split (80% Train, 20% Val)
indices = np.arange(len(train_df))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

X_train = {
    'title_input': title_seq[train_idx],
    'content_input': content_seq[train_idx],
    'tabular_input': tabular_features[train_idx]
}
y_train = labels[train_idx]

X_val = {
    'title_input': title_seq[val_idx],
    'content_input': content_seq[val_idx],
    'tabular_input': tabular_features[val_idx]
}
y_val = labels[val_idx]

print(f"Train: {len(y_train)}, Val: {len(y_val)}")
print(f"Train label dist: {np.bincount(y_train.astype(int))}")
print(f"Val label dist:   {np.bincount(y_val.astype(int))}")


# 8. Model Architecture (BiGRU + Handcrafted Features)
Menggabungkan urutan teks dengan metrik kedekatan statistik (BM25, TF-IDF).

In [ ]:
def build_model(gru_units=64, dropout_rate=0.3, learning_rate=1e-3):
    # 1. Inputs
    title_in = Input(shape=(MAX_TITLE_LEN,), name='title_input')
    content_in = Input(shape=(MAX_CONTENT_LEN,), name='content_input')
    tabular_in = Input(shape=(4,), name='tabular_input') # 4 custom features
    
    # 2. Embedding Layer (Dilatih dari awal sesuai rules)
    embed_layer = Embedding(input_dim=MAX_VOCAB, output_dim=128, name='word_embedding')
    
    # 3. RNN Processing (BiGRU)
    title_embed = embed_layer(title_in)
    content_embed = embed_layer(content_in)
    
    # Shared GRU architecture for semantic extraction
    shared_gru = Bidirectional(GRU(gru_units, return_sequences=True))
    
    title_gru = GlobalMaxPooling1D()(shared_gru(title_embed))
    content_gru = GlobalMaxPooling1D()(shared_gru(content_embed))
    
    # 4. Concatenate semuanya (Wide + Deep)
    merged = Concatenate()([title_gru, content_gru, tabular_in])
    
    # 5. Fully Connected Layers
    dense_1 = Dense(128, activation='relu')(merged)
    dense_1 = Dropout(dropout_rate)(dense_1)
    dense_2 = Dense(64, activation='relu')(dense_1)
    
    # 6. Output Layer (Biner)
    output = Dense(1, activation='sigmoid', name='output')(dense_2)
    
    model = Model(inputs=[title_in, content_in, tabular_in], outputs=output)
    
    # Compile model
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

# Contoh Build 
dummy_model = build_model()
dummy_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title_input         │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_input       │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 300, 128)  │  3,200,000 │ title_input[0][0… │
│ (Embedding)         │                   │            │ content_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 300, 128)  │     74,496 │ word_embedding[0… │
│ (Bidirectional)     │                   │            │ word_embedding[1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ bidirectional[0]… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ bidirectional[1]… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tabular_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 260)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ tabular_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     33,408 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         65 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,316,225 (12.65 MB)

 Trainable params: 3,316,225 (12.65 MB)

 Non-trainable params: 0 (0.00 B)

# 9. Hyperparameter Optimization (HPO) menggunakan Optuna
Mencari parameter GRU Units, Dropout, dan Learning Rate terbaik.

In [ ]:
DO_HPO = True

def objective(trial):
    gru_units = trial.suggest_categorical('gru_units', [32, 64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 5e-3, log=True)
    
    model = build_model(gru_units=gru_units, dropout_rate=dropout_rate, learning_rate=learning_rate)
    
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=5,
        batch_size=64,
        verbose=0
    )
    
    y_pred_prob = model.predict(X_val, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    macro_f1 = f1_score(y_val, y_pred, average='macro')
    return macro_f1

if DO_HPO:
    print("Memulai Hyperparameter Optimization...")
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)
    
    print(f"\nParameter Terbaik: {study.best_params}")
    print(f"Macro F1 Terbaik (Validasi): {study.best_value:.4f}")
    
    best_model = build_model(**study.best_params)
else:
    best_model = build_model(gru_units=64, dropout_rate=0.3, learning_rate=1e-3)

print("\nMelatih Model Final (lebih banyak epoch)...")
history = best_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)


[I 2026-09-09 23:06:40,990] A new study created in memory with name: no-name-8db762b4-aff8-4967-8a6c-44dbb4c9adea


Memulai Hyperparameter Optimization...
153/153 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step


[I 2026-09-09 23:09:43,331] Trial 0 finished with value: 0.9063905966756376 and parameters: {'gru_units': 32, 'dropout_rate': 0.2656170369445294, 'learning_rate': 0.001736346784756297}. Best is trial 0 with value: 0.9063905966756376.


153/153 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step


[I 2026-09-09 23:13:12,511] Trial 1 finished with value: 0.8940783269170836 and parameters: {'gru_units': 32, 'dropout_rate': 0.4358826728910311, 'learning_rate': 0.00018296073944471552}. Best is trial 0 with value: 0.9063905966756376.


Parameter Terbaik:  {'gru_units': 32, 'dropout_rate': 0.2656170369445294, 'learning_rate': 0.001736346784756297}
Macro F1 Terbaik (Validasi):  0.9063905966756376

Melatih Model Final...
Epoch 1/5
305/305 ━━━━━━━━━━━━━━━━━━━━ 138s 419ms/step - accuracy: 0.8131 - loss: 0.6396 - val_accuracy: 0.8801 - val_loss: 0.3324
Epoch 2/5
305/305 ━━━━━━━━━━━━━━━━━━━━ 100s 327ms/step - accuracy: 0.9242 - loss: 0.2458 - val_accuracy: 0.9328 - val_loss: 0.2306
Epoch 3/5
305/305 ━━━━━━━━━━━━━━━━━━━━ 122s 400ms/step - accuracy: 0.9329 - loss: 0.2083 - val_accuracy: 0.9170 - val_loss: 0.2426
Epoch 4/5
305/305 ━━━━━━━━━━━━━━━━━━━━ 125s 409ms/step - accuracy: 0.9429 - loss: 0.1626 - val_accuracy: 0.9279 - val_loss: 0.2454
Epoch 5/5
305/305 ━━━━━━━━━━━━━━━━━━━━ 107s 352ms/step - accuracy: 0.9548 - loss: 0.1236 - val_accuracy: 0.8955 - val_loss: 0.3515


# 10. Evaluasi Akhir (Macro F1-Score)

In [ ]:
print("=== HASIL EVALUASI VALIDASI ===")
y_pred_prob = best_model.predict(X_val)
y_pred = (y_pred_prob > 0.5).astype(int)

macro_f1 = f1_score(y_val, y_pred, average='macro')
print(f"Macro F1-Score: {macro_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['Tidak Sesuai (0)', 'Sesuai (1)']))

=== HASIL EVALUASI VALIDASI ===
153/153 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step
Macro F1-Score: 0.8899

Classification Report:
                  precision    recall  f1-score   support

Tidak Sesuai (0)       0.86      0.87      0.87      1887
      Sesuai (1)       0.92      0.91      0.91      2992

        accuracy                           0.90      4879
       macro avg       0.89      0.89      0.89      4879
    weighted avg       0.90      0.90      0.90      4879



In [ ]:
import pandas as pd
import numpy as np

print("=== MEMULAI PREDIKSI DATA TEST ===")

# 1. Load Data Test
test_df = pd.read_csv('data/test.csv')
try:
    sample_sub = pd.read_csv('data/sample_submission.csv')
except FileNotFoundError:
    sample_sub = pd.DataFrame({'id': test_df['id']})

# 2. Preprocessing
test_df['title'] = test_df['title'].fillna('').astype(str)
test_df['content'] = test_df['content'].fillna('').astype(str)

print("Preprocessing teks test...")
test_df['clean_title'] = test_df['title'].apply(preprocess_text)
test_df['clean_content'] = test_df['content'].apply(preprocess_text)

# 3. Feature Engineering (pakai corpus train untuk TF-IDF fit)
test_df, _ = extract_features(test_df, full_corpus_title=train_corpus_title, full_corpus_content=train_corpus_content)

# 4. Tokenisasi untuk RNN
print("Konversi ke sequence...")
test_title_seq = pad_sequences(tokenizer.texts_to_sequences(test_df['clean_title']), maxlen=MAX_TITLE_LEN, padding='post')
test_content_seq = pad_sequences(tokenizer.texts_to_sequences(test_df['clean_content']), maxlen=MAX_CONTENT_LEN, padding='post')
test_tabular_features = test_df[['jaccard_sim', 'tfidf_cosine_sim', 'bm25_score', 'len_diff']].values

X_test = {
    'title_input': test_title_seq,
    'content_input': test_content_seq,
    'tabular_input': test_tabular_features
}

# 5. Prediksi
print("Menjalankan prediksi model...")
test_preds_probs = best_model.predict(X_test)

# 6. Cari Threshold Terbaik dari Validation (TANPA BUG)
print("Mencari threshold terbaik...")
val_preds_probs = best_model.predict(X_val)

best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.1, 0.9, 0.01):
    val_preds = (val_preds_probs >= th).astype(int)
    f1 = f1_score(y_val, val_preds, average='macro')
    if f1 > best_f1:
        best_f1 = f1
        best_th = th

print(f"Threshold optimal: {best_th:.2f} | Val Macro F1: {best_f1:.4f}")

# 7. Apply threshold ke test
test_labels = (test_preds_probs >= best_th).astype(int)
print(f"\nDistribusi prediksi test: {np.bincount(test_labels.flatten().astype(int))}")


=== MEMULAI PREDIKSI DATA TEST ===
Preprocessing teks test...
Mengekstrak Fitur Kemiripan Teks...
Feature Engineering Selesai.
Konversi ke sequence...
Menjalankan prediksi model...
113/113 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step
Mencari threshold terbaik di set validasi...
153/153 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step
Threshold optimal ditemukan: 0.10 dengan Val F1: 0.9158


## 11. Generate Submission

In [ ]:
import os

sample_sub['label'] = test_labels.flatten().astype(int)
sample_sub.to_csv('submission.csv', index=False)

print(f'Submission saved! Threshold={best_th:.2f}, Val F1={best_f1:.4f}')
print(f'Distribusi prediksi:')
print(sample_sub['label'].value_counts())
print(f"\nFile: {os.path.abspath('submission.csv')}")


Submission saved! Threshold=0.10, F1=0.9158
Distribusi prediksi:
label
1    3528
0      75
Name: count, dtype: int64

File submission.csv berhasil dibuat dan bisa kamu temukan di:
c:\Users\Ezra Faira\Documents\1. KULIAH\Lomba\submission.csv
